In [ ]:
import os
from PIL import Image, ImageFile
from pathlib import Path
import numpy as np

# Allow PIL to load truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

def convert_tif_to_jpg(input_folder, output_folder, quality=100):
    """
    Convert 16-bit TIFF to JPEG.
    Method: linear scaling, maps original pixel range to 0-255.
    Equivalent to checking "Scale When Converting" in ImageJ —
    image brightness will look normal and not too dark.
    """
    
    # Create output folder if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    # Get all .tif and .tiff files from input folder
    tif_files = []
    for ext in ['*.tif', '*.tiff', '*.TIF', '*.TIFF']:
        tif_files.extend(Path(input_folder).glob(ext))
    
    if not tif_files:
        print(f"No TIFF files found in {input_folder}")
        return
    
    print(f"Found {len(tif_files)} TIFF file(s)")
    
    # Convert each TIFF file to JPEG
    for tif_path in tif_files:
        try:
            print(f"\nProcessing: {tif_path.name}")
            
            # Open TIFF image
            with Image.open(tif_path) as img:
                print(f"  Original mode: {img.mode}, Size: {img.size}")
                
                # Handle multi-page TIFFs - use first frame
                if hasattr(img, 'n_frames') and img.n_frames > 1:
                    print(f"  Multi-page TIFF with {img.n_frames} frames, using first frame")
                    img.seek(0)
                
                # Convert to numpy array
                img_array = np.array(img)
                original_min = img_array.min()
                original_max = img_array.max()
                print(f"  Original dtype: {img_array.dtype}, range: [{original_min}, {original_max}]")
                
                # For 16-bit: linear scaling to 0-255
                if img_array.dtype == np.uint16:
                    print(f"  Converting 16-bit to 8-bit with linear scaling")
                    print(f"  Mapping [{original_min}, {original_max}] -> [0, 255]")
                    
                    if original_max > original_min:
                        # Scale to 0-255
                        scaled = (img_array.astype(np.float32) - original_min) / (original_max - original_min) * 255
                        img_array = scaled.astype(np.uint8)
                    else:
                        # All pixels same value
                        img_array = np.zeros_like(img_array, dtype=np.uint8) if original_min == 0 else np.full_like(img_array, 255, dtype=np.uint8)
                    
                    img = Image.fromarray(img_array)
                    print(f"  Output range: [{img_array.min()}, {img_array.max()}]")
                
                # For 8-bit: no conversion needed
                elif img_array.dtype == np.uint8:
                    print(f"  Already 8-bit, no conversion needed")
                
                # Convert grayscale to RGB (JPEG requires RGB)
                if img.mode == 'L':
                    print(f"  Converting grayscale to RGB")
                    img = img.convert('RGB')
                
                # Create output filename
                output_filename = tif_path.stem + '.jpg'
                output_path = Path(output_folder) / output_filename

                # Save as JPEG
                img.save(output_path, 'JPEG', quality=quality, optimize=False)
                print(f"  ✓ Successfully saved: {output_filename}")
                
        except Exception as e:
            print(f"  ✗ Error converting {tif_path.name}: {str(e)}")
            import traceback
            traceback.print_exc()
    
    print("\n" + "="*50)
    print("Conversion complete!")
    print(f"Output saved to: {output_folder}")
    print("="*50)


# Example usage
if __name__ == "__main__":
    input_folder = "/Users/daizongsun/Desktop/DLBCL DL Project/Processed DLBCL/05-05-2026 DLBCL 122851/formatted_cd45ra/"
    output_folder = "/Users/daizongsun/Desktop/DLBCL DL Project/Processed DLBCL/05-05-2026 DLBCL 122851/formatted_cd45ra_jpg/"
    
    convert_tif_to_jpg(input_folder, output_folder, quality=100)